# Pelatihan Model Prediksi Sisa Umur Mesin (NASA Turbofan RUL)

Notebook ini melatih dua jenis model untuk menebak sisa umur mesin pesawat:
1. Model Cepat: LightGBM (berbasis pohon keputusan)
2. Model Rangkaian Waktu: PyTorch CNN + BiGRU (berbasis urutan waktu)

Data yang dipakai berasal dari dbt yang sudah bersih dan siap pakai.

## 1. Persiapan Lingkungan dan Instalasi Paket

In [ ]:
!pip install lightgbm optuna onnx onnxruntime pyarrow matplotlib -q

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Menggunakan alat:', device)

## 2. Membaca Data Fitur

Kita memuat file parquet yang dihasilkan dari proses dbt sebelumnya.

In [ ]:
df = pd.read_parquet('features_fd001.parquet')
print('Ukuran data:', df.shape)
df.head()

## 3. Pemeriksaan Data Singkat (EDA)

Data tidak perlu dibersihkan lagi karena sensor yang mati dan nilai kosong sudah ditangani saat transformasi data di dbt.
Di sini kita hanya memastikan persebaran sisa umur dan pola sensor mesin.

In [ ]:
print('Jumlah mesin pesawat:', df['engine_id'].nunique())
print('Nilai kosong per kolom:')
print(df.isnull().sum().max())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df['target_rul'], bins=20, color='skyblue', edgecolor='black')
plt.title('Penyebaran Target RUL')
plt.xlabel('Sisa Siklus')
plt.ylabel('Frekuensi')

sample_engine = df[df['engine_id'] == 1]
plt.subplot(1, 2, 2)
plt.plot(sample_engine['cycle'], sample_engine['sensor_2_lpc_temp'], color='red')
plt.title('Pola Suhu Mesin #1 Terhadap Siklus')
plt.xlabel('Siklus')
plt.ylabel('Suhu')
plt.tight_layout()
plt.show()

## 4. Pembagian Data Uji dan Latih Berdasarkan Nomor Mesin

Kita membagi data berdasarkan nomor mesin, bukan acak per baris. 
Tujuannya agar model tidak menghafal data dari mesin yang sama saat diuji.

In [ ]:
feature_cols = [
    col for col in df.columns 
    if col not in ['engine_id', 'cycle', 'max_cycle', 'raw_rul', 'target_rul', 'event_timestamp', 'ingested_at']
]
print('Jumlah kolom fitur:', len(feature_cols))

train_engines = df['engine_id'].unique()[:80]
val_engines = df['engine_id'].unique()[80:]

train_df = df[df['engine_id'].isin(train_engines)].copy()
val_df = df[df['engine_id'].isin(val_engines)].copy()

X_train = train_df[feature_cols]
y_train = train_df['target_rul']
X_val = val_df[feature_cols]
y_val = val_df['target_rul']

print('Baris data latih:', len(train_df))
print('Baris data validasi:', len(val_df))

## 5. Rumus Penilaian Khusus NASA

Dalam dunia pesawat, terlambat menebak kerusakan mesin jauh lebih bahaya dibanding menebak terlalu dini. 
Rumus ini memberi hukuman nilai yang jauh lebih berat jika model terlambat mendeteksi kerusakan.

In [ ]:
def calculate_nasa_score(y_true, y_pred):
    diff = y_pred - y_true
    score = 0.0
    for d in diff:
        if d < 0:
            score += (math.exp(-d / 13.0) - 1.0)
        else:
            score += (math.exp(d / 10.0) - 1.0)
    return score

def evaluate_predictions(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    nasa = calculate_nasa_score(y_true.values, y_pred)
    print(f'Hasil Model: {name}')
    print(f'RMSE : {rmse:.2f} siklus')
    print(f'MAE  : {mae:.2f} siklus')
    print(f'Skor NASA : {nasa:.2f}')
    return {'model': name, 'rmse': rmse, 'mae': mae, 'nasa_score': nasa}

## 6. Model 1: LightGBM dan Penyetelan Parameter Otomatis (Tuning)

Mengapa butuh tuning?
Pohon keputusan bisa membuat model terlalu pintar menghafal data latih dan gagal saat membaca mesin baru.
Kita mencari kombinasi kedalaman pohon dan kecepatan belajar terbaik menggunakan Optuna.

In [ ]:
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)
print('Parameter Terbaik LightGBM:', study.best_params)

In [ ]:
best_lgb = lgb.LGBMRegressor(**study.best_params, random_state=42, verbosity=-1)
best_lgb.fit(X_train, y_train)

lgb_val_preds = best_lgb.predict(X_val)
metrics_lgb = evaluate_predictions('LightGBM (Tuned)', y_val, lgb_val_preds)

## 7. Model 2: Deep Learning Berbasis Rangkaian Waktu (PyTorch CNN + BiGRU)

Mengapa model ini dipakai?
Mesin tidak rusak tiba-tiba dalam satu siklus, melainkan ada proses keausan berantai.
Model ini membaca 30 siklus terakhir secara berurutan layaknya membaca video.

In [ ]:
scaler = StandardScaler()
scaled_train = scaler.fit_transform(X_train)
scaled_val = scaler.transform(X_val)

train_df_scaled = train_df.copy()
val_df_scaled = val_df.copy()
train_df_scaled[feature_cols] = scaled_train
val_df_scaled[feature_cols] = scaled_val

def create_sliding_windows(data, feature_names, seq_len=30):
    sequences = []
    targets = []
    for engine in data['engine_id'].unique():
        eng_df = data[data['engine_id'] == engine].sort_values('cycle')
        feat_vals = eng_df[feature_names].values
        target_vals = eng_df['target_rul'].values
        
        if len(eng_df) < seq_len:
            pad = np.repeat(feat_vals[0:1, :], seq_len - len(eng_df), axis=0)
            feat_vals = np.vstack([pad, feat_vals])
            target_vals = np.concatenate([np.repeat(target_vals[0], seq_len - len(eng_df)), target_vals])
            
        for i in range(len(feat_vals) - seq_len + 1):
            sequences.append(feat_vals[i:i + seq_len])
            targets.append(target_vals[i + seq_len - 1])
            
    return np.array(sequences, dtype=np.float32), np.array(targets, dtype=np.float32)

X_train_seq, y_train_seq = create_sliding_windows(train_df_scaled, feature_cols, seq_len=30)
X_val_seq, y_val_seq = create_sliding_windows(val_df_scaled, feature_cols, seq_len=30)

print('Bentuk data rangkaian latih:', X_train_seq.shape)
print('Bentuk data rangkaian validasi:', X_val_seq.shape)

In [ ]:
class TurbofanDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.sequences)
        
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

train_loader = DataLoader(TurbofanDataset(X_train_seq, y_train_seq), batch_size=64, shuffle=True)
val_loader = DataLoader(TurbofanDataset(X_val_seq, y_val_seq), batch_size=64, shuffle=False)

In [ ]:
class TurbofanBiGRU(nn.Module):
    def __init__(self, num_features, hidden_dim=64, num_layers=2):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(input_size=32, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        x_trans = x.transpose(1, 2)
        conv_out = self.relu(self.conv1(x_trans))
        conv_out = conv_out.transpose(1, 2)
        gru_out, _ = self.gru(conv_out)
        last_hidden = gru_out[:, -1, :]
        out = self.fc(last_hidden)
        return out

deep_model = TurbofanBiGRU(num_features=len(feature_cols)).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(deep_model.parameters(), lr=0.001, weight_decay=1e-4)

In [ ]:
epochs = 20
best_val_loss = float('inf')

for epoch in range(1, epochs + 1):
    deep_model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = deep_model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(batch_x)
        
    train_loss /= len(train_loader.dataset)
    
    deep_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            output = deep_model(batch_x)
            loss = criterion(output, batch_y)
            val_loss += loss.item() * len(batch_x)
    val_loss /= len(val_loader.dataset)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(deep_model.state_dict(), 'best_bigru_model.pt')
        
    if epoch % 5 == 0 or epoch == 1:
        print(f'Putaran {epoch:02d}/{epochs} - Rugi Latih: {math.sqrt(train_loss):.2f} - Rugi Validasi: {math.sqrt(val_loss):.2f}')

## 8. Evaluasi Model Deep Learning dan Perbandingan Akhir

In [ ]:
deep_model.load_state_dict(torch.load('best_bigru_model.pt'))
deep_model.eval()

with torch.no_grad():
    test_x = torch.tensor(X_val_seq, dtype=torch.float32).to(device)
    deep_preds = deep_model(test_x).cpu().numpy().flatten()

metrics_deep = evaluate_predictions('PyTorch CNN-BiGRU', pd.Series(y_val_seq), deep_preds)

In [ ]:
summary = pd.DataFrame([metrics_lgb, metrics_deep])
display(summary)

plt.figure(figsize=(10, 4))
plt.plot(y_val_seq[:150], label='Nilai Asli', color='black', linewidth=2)
plt.plot(lgb_val_preds[:150], label='Tebakan LightGBM', color='orange', alpha=0.8)
plt.plot(deep_preds[:150], label='Tebakan CNN-BiGRU', color='blue', alpha=0.8)
plt.title('Perbandingan Tebakan Sisa Umur (150 Data Pertama)')
plt.xlabel('Urutan Sampel')
plt.ylabel('Sisa Umur (Siklus)')
plt.legend()
plt.show()

## 9. Simpan Model Siap Pakai ke Format Standar (ONNX)

Format ONNX membuat model bisa dijalankan dengan sangat cepat di komputer server atau laptop tanpa perlu kartu grafis.

In [ ]:
dummy_input = torch.randn(1, 30, len(feature_cols), dtype=torch.float32).to(device)
onnx_file_path = 'best_rul_model.onnx'

torch.onnx.export(
    deep_model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input_sequence'],
    output_names=['predicted_rul'],
    dynamic_axes={
        'input_sequence': {0: 'batch_size'},
        'predicted_rul': {0: 'batch_size'}
    }
)

print('Model berhasil disimpan ke format ONNX:', onnx_file_path)